<a href="https://colab.research.google.com/github/Pauaza/PBL_Sistem-Cerdas-Rencana-Riset/blob/uts-sbp/Forward_Chaining.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import re
import io
from google.colab import files
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# ==========================================
# 1. LOAD FILE DATASET
# ==========================================
df_master = pd.read_csv('master_dataset_dosen.csv')
d_master = df.fillna('')

# Preprocessing awal: Mengisi nilai kosong agar tidak error saat pemrosesan teks
df_master['lab_name'] = df_master['lab_name'].fillna('')
df_master['judul_penelitian'] = df_master['judul_penelitian'].fillna('')
df_master['judul_skripsi'] = df_master['judul_skripsi'].fillna('')

In [ ]:
# ==========================================
# 2. DEFINISI FUNGSI GLOBAL
# ==========================================

def calculate_sim(column_data, query, vectorizer):
    """
    Fungsi untuk menghitung kemiripan antara input user dengan kolom dataset
    """
    corpus = column_data.tolist() + [query]
    tfidf_matrix = vectorizer.fit_transform(corpus)
    # Menghitung similarity antara baris terakhir (input) dengan semua baris dataset
    return cosine_similarity(tfidf_matrix[-1], tfidf_matrix[:-1])[0]

def sistem_rekomendasi_skripsi(df, input_lab, input_deskripsi):
    """
    Fungsi utama menggunakan logika Forward Chaining dan pembobotan
    """
    # Inisialisasi Vectorizer
    vectorizer = TfidfVectorizer(stop_words='english')
    user_input_combined = f"{input_lab} {input_deskripsi}"

    # --- PROSES FORWARD CHAINING (RULE MATCHING) ---

    # Rule 1: Kecocokan Judul Penelitian (Bobot 50%)
    score_penelitian = calculate_sim(df['judul_penelitian'], user_input_combined, vectorizer)

    # Rule 2: Kecocokan Judul Skripsi (Bobot 30%)
    score_skripsi = calculate_sim(df['judul_skripsi'], user_input_combined, vectorizer)

    # Rule 3: Kecocokan Nama Lab (Bobot 20%)
    score_lab = calculate_sim(df['lab_name'], input_lab, vectorizer)

    # --- INFERENSI (HITUNG SKOR TOTAL) ---
    # Membuat salinan agar tidak merubah dataframe asli secara permanen
    results_df = df.copy()
    results_df['match_score'] = (
        (0.50 * score_penelitian) +
        (0.30 * score_skripsi) +
        (0.20 * score_lab)
    ) * 100

    # --- GENERATE JUDUL (AI PATTERN) ---
    clean_ide = re.sub(r'[^\w\s]', '', input_deskripsi)
    words = clean_ide.split()
    objek = words[0] if len(words) > 0 else "Sistem"

    ai_titles = [
        f"Implementasi {input_lab} untuk Otomasi {input_deskripsi.split('pada')[0] if 'pada' in input_deskripsi else input_deskripsi}",
        f"Rancang Bangun Sistem {objek} Berbasis {input_lab} dengan Integrasi Sensor Terdistribusi",
        f"Analisis Performa Algoritma Cerdas pada {input_deskripsi} Berbasis {input_lab}",
        f"Pengembangan Dashboard Real-time {input_lab} untuk Optimasi Sistem {objek}",
        f"Penerapan Metode Forward Chaining dalam Klasifikasi Data pada {input_deskripsi}"
    ]

    # Mengambil Top 5 Dosen
    top_5_dosen = results_df.sort_values(by='match_score', ascending=False).head(5)

    return top_5_dosen, ai_titles

In [ ]:
# ==========================================
# 3. ANTARMUKA INPUT & OUTPUT
# ==========================================

print("\n" + "="*30)
print("SISTEM REKOMENDASI SKRIPSI")
print("="*30)

# Input dari pengguna
input_user_lab = input("Masukkan Lab/Topik (contoh: Internet of Things): ")
input_user_ide = input("Masukkan Deskripsi Ide Skripsi: ")

# Menjalankan Sistem
rekomendasi_dosen, judul_ai = sistem_rekomendasi_skripsi(df_master, input_user_lab, input_user_ide)

# Output Judul
print("\n" + "="*40)
print("1. REKOMENDASI JUDUL SKRIPSI (GENERATED AI)")
print("="*40)
for i, judul in enumerate(judul_ai, 1):
    print(f"{i}. {judul}")

# Output Dosen
print("\n" + "="*40)
print("2. TOP 5 REKOMENDASI DOSEN PEMBIMBING")
print("="*40)
for index, row in rekomendasi_dosen.iterrows():
    print(f"[{row['match_score']:.2f}%] - {row['nama_dosen']}")
    print(f"       Lab: {row['lab_name']}")
    print("-" * 20)


SISTEM REKOMENDASI SKRIPSI
Masukkan Lab/Topik (contoh: Internet of Things): Business Analyst
Masukkan Deskripsi Ide Skripsi: Pengembangan sistem prediksi stok barang otomatis untuk toko retail

1. REKOMENDASI JUDUL SKRIPSI (GENERATED AI)
1. Implementasi Business Analyst untuk Otomasi Pengembangan sistem prediksi stok barang otomatis untuk toko retail
2. Rancang Bangun Sistem Pengembangan Berbasis Business Analyst dengan Integrasi Sensor Terdistribusi
3. Analisis Performa Algoritma Cerdas pada Pengembangan sistem prediksi stok barang otomatis untuk toko retail Berbasis Business Analyst
4. Pengembangan Dashboard Real-time Business Analyst untuk Optimasi Sistem Pengembangan
5. Penerapan Metode Forward Chaining dalam Klasifikasi Data pada Pengembangan sistem prediksi stok barang otomatis untuk toko retail

2. TOP 5 REKOMENDASI DOSEN PEMBIMBING
[12.42%] - Candra Bella Vista, S.Kom., MT.
       Lab: Business Analytics
--------------------
[12.08%] - Ir. Rudy Ariyanto, ST., M.Cs.
       Lab: